<a href="https://colab.research.google.com/github/EsarFatima/MachineLearning-flyrank-/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.1 MB/s eta 0:00:00


In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/EsarFatima/MachineLearning-flyrank-"
REPO_DIR = "MachineLearning-flyrank-"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np
from pypdf import PdfReader

# Read the paper's own text directly rather than relying on memory of it.
reader = PdfReader("docs/flyrank-seo-research-march-2026.pdf")
print(f"{len(reader.pages)}-page report loaded.\n")

for label, page in [("Finding A -- 'What Predicts Health?'", 26),
                     ("Finding B -- 'What Predicts Growth?'", 28)]:
    text = reader.pages[page].extract_text().replace("\n", " ")
    print(f"--- {label} (page {page+1}) ---")
    print(text[:260])
    print()

36-page report loaded.

--- Finding A -- 'What Predicts Health?' (page 27) ---
FlyRank ML APPENDIX — FEATURE IMPORTANCE What Predicts Health? Random Forest feature importance for predicting health score. The model is holdout-tested, but the target  itself is partly constructed from some of these inputs, so importance is descriptive rathe

--- Finding B -- 'What Predicts Growth?' (page 29) ---
FlyRank ML APPENDIX — GROWTH & CLASSIFICATION What Predicts Growth? Logistic regression (71% holdout accuracy) describing which sampled features separate growing from  declining pages. GROWTH PREDICTION COEFFICIENTS (LOGISTIC REGRESSION) Content Age 1 Days Sin



**Finding A — "What Predicts Health?" (Random Forest → `health_score`, p.27).** The report's own methodology section defines `health_score` as a hand-built composite: impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts). The Random Forest here is asked to predict that composite using — among other inputs — average position, impressions, CTR, and scroll depth, which is where three of the top four "predictors" (Average Position 43%, Impressions 32%, Scroll Depth 15%) come from. That's the exact Type-1 leakage shape our own w03 leakage hunt tested for on `is_declining_label`: a label built from a formula, with pieces of that same formula fed back in as features. To the paper's credit, the text says this out loud — "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal." **My methodology question, framed as a strengthening suggestion:** an 80/20 holdout split (per the Methodology page) protects against overfitting to noise, but it can't rescue a label that is a deterministic formula of its own features — no split design fixes that. The finding would be strictly stronger if it either (a) re-ran the importance ranking with position/impressions/CTR/scroll-depth excluded, showing what — if anything — the *other* six features (clicks, sessions, age, word count, days visible, AI sessions) contribute, or (b) framed it plainly as "confirming the known formula's weights show up in the fit," not as a discovery about health at all.

**Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.29).** The label here is growing vs. declining, built the same way ours is — from a 30-day-vs-prior-30-day impression trend. Two methodology questions, both constructive: **First**, the Methodology page states an 80/20 split for Random Forest, Logistic Regression, and the Decision Tree, but doesn't say whether that split is grouped by brand. The dataset spans 57 brands over 61.8K rows — the same shape as our own 32-clients-over-30K-rows dataset, where I measured below (Section 2) that a plain row-level split inflates Random Forest AUC by +0.131 over an honest client-grouped split, purely from brand-level memorization. If Finding B's split isn't grouped by brand, the same inflation risk applies, and the fix is cheap: re-run it grouped and report both numbers side by side, the way Section 2 does here. **Second**, 71% holdout accuracy is reported with no base rate next to it anywhere on that page or in the Methodology section — the honest-claims skill flags this exact gap ("accuracy without its base rate next to it" is a banned pattern) because a 71%-accurate model on a near-50/50 label is a very different finding than 71% on an 90/10 label. Both fixes are small additions to a finding that's already labeled exploratory and appropriately hedged elsewhere in the paper — this is about making a good-faith finding fully checkable, not undermining it.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])

num_features = ["content_age_days", "days_since_last_update", "log_impressions_90d",
                 "avg_position", "ctr", "engagement_rate", "search_volume", "competition",
                 "word_count", "has_keyword_data", "has_word_count"]
cat_features = ["content_type", "main_intent"]

X_num = df[num_features].replace([np.inf, -np.inf], np.nan).fillna(0)
X_cat = pd.get_dummies(df[cat_features].fillna("unknown"), drop_first=True)
X = pd.concat([X_num, X_cat], axis=1)
y = df["is_declining_label"]
groups = df["client_id"]

def fit_eval(Xtr, Xte, ytr, yte):
    scaler = StandardScaler()
    Xtr_s, Xte_s = scaler.fit_transform(Xtr), scaler.transform(Xte)
    logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    logreg.fit(Xtr_s, ytr)
    logreg_auc = roc_auc_score(yte, logreg.predict_proba(Xte_s)[:, 1])
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                                 class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(Xtr, ytr)
    rf_auc = roc_auc_score(yte, rf.predict_proba(Xte)[:, 1])
    return logreg_auc, rf_auc

# BEFORE -- plain row-level random 80/20 split (the split a first pass tends to reach for)
Xtr_r, Xte_r, ytr_r, yte_r, gtr_r, gte_r = train_test_split(
    X, y, groups, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
overlap_random = set(gtr_r) & set(gte_r)
logreg_auc_r, rf_auc_r = fit_eval(Xtr_r, Xte_r, ytr_r, yte_r)

# AFTER -- grouped 80/20 split by client_id, same as w05 (the honest one)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))
Xtr_g, Xte_g = X.iloc[train_idx], X.iloc[test_idx]
ytr_g, yte_g = y.iloc[train_idx], y.iloc[test_idx]
overlap_grouped = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
logreg_auc_g, rf_auc_g = fit_eval(Xtr_g, Xte_g, ytr_g, yte_g)

print("=== BEFORE: random row-level 80/20 split ===")
print("client overlap between train/test:", len(overlap_random), "of", groups.nunique(), "total clients")
print(f"Logistic Regression AUC: {logreg_auc_r:.3f}")
print(f"Random Forest AUC:       {rf_auc_r:.3f}")
print()
print("=== AFTER: grouped 80/20 split by client_id (honest) ===")
print("client overlap between train/test:", len(overlap_grouped), "(must be 0)")
print(f"Logistic Regression AUC: {logreg_auc_g:.3f}")
print(f"Random Forest AUC:       {rf_auc_g:.3f}")
print()
print("Gap (random minus grouped) -- this gap IS the memorization, not extra skill:")
print(f"  Logistic Regression: {logreg_auc_r - logreg_auc_g:+.3f}")
print(f"  Random Forest:       {rf_auc_r - rf_auc_g:+.3f}")

=== BEFORE: random row-level 80/20 split ===
client overlap between train/test: 31 of 32 total clients
Logistic Regression AUC: 0.646
Random Forest AUC:       0.723

=== AFTER: grouped 80/20 split by client_id (honest) ===
client overlap between train/test: 0 (must be 0)
Logistic Regression AUC: 0.541
Random Forest AUC:       0.591

Gap (random minus grouped) -- this gap IS the memorization, not extra skill:
  Logistic Regression: +0.104
  Random Forest:       +0.131


**Reading the gap.** 31 of 32 clients show up on both sides of the random split — with only 32 clients and 30K rows, a row-level split was never going to separate them. That overlap is exactly what lets Random Forest AUC read 0.723 instead of its honest 0.591: +0.131 of "skill" that's actually the model recognizing clients it already saw in training, not predicting decline for a client it hasn't. Logistic Regression shows the same pattern at a smaller scale (+0.104). This is the same failure mode flagged as a methodology question for Finding B above — now measured directly on data with the same shape (many rows per group, a modest number of groups). Every AUC I report from here forward uses the grouped split; the random-split numbers exist only to show the size of the gap.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Re-run the w03 confession test on the FINAL feature set (unchanged since w03), on the honest
# grouped split defined above -- confirming no leakage crept in while w04/w05 were built.
def make_X(extra_cols):
    num = num_features + extra_cols
    Xn = df[num].replace([np.inf, -np.inf], np.nan).fillna(0)
    Xc = pd.get_dummies(df[cat_features].fillna("unknown"), drop_first=True)
    return pd.concat([Xn, Xc], axis=1)

for label, extra in [("Final feature set (clean, unchanged since w03)", []),
                      ("WITH impressions_last_30d + impressions_prev_30d", ["impressions_last_30d", "impressions_prev_30d"]),
                      ("WITH trend_pct itself (worst case)", ["trend_pct"])]:
    Xc = make_X(extra)
    Xtr_c, Xte_c = Xc.iloc[train_idx], Xc.iloc[test_idx]
    scaler = StandardScaler()
    Xtr_s, Xte_s = scaler.fit_transform(Xtr_c), scaler.transform(Xte_c)
    m = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    m.fit(Xtr_s, ytr_g)
    auc = roc_auc_score(yte_g, m.predict_proba(Xte_s)[:, 1])
    print(f"  {label}: AUC = {auc:.3f}")

print()
print("No product/health-score flags exist in this starter file at all (data-dictionary.md),")
print("so none could leak in here either -- same conclusion as w03, re-checked on the final set.")
print("Final approved feature set:", num_features + cat_features)

  Final feature set (clean, unchanged since w03): AUC = 0.541
  WITH impressions_last_30d + impressions_prev_30d: AUC = 0.854
  WITH trend_pct itself (worst case): AUC = 1.000

No product/health-score flags exist in this starter file at all (data-dictionary.md),
so none could leak in here either -- same conclusion as w03, re-checked on the final set.
Final approved feature set: ['content_age_days', 'days_since_last_update', 'log_impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'search_volume', 'competition', 'word_count', 'has_keyword_data', 'has_word_count', 'content_type', 'main_intent']


Re-run on the honest grouped split, the confession test still confirms the w03 finding: the clean feature set holds at AUC 0.541, adding the two raw 30-day windows jumps it to 0.854, and adding `trend_pct` itself hits a perfect 1.000. (These exact numbers move slightly from w03's, because that hunt used a plain train/test split rather than this notebook's client-grouped one — the *conclusion* doesn't move: all four label-derived siblings stay out of the final feature set, and no product flags exist in this file to leak in the first place.)

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# Precision@K on the SAME honest grouped test set, to ground the rewrite in real numbers.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

stale = (df["days_since_last_update"] >= 90).astype(int)
visible = ((df["impressions_90d"] >= 300) & (df["impressions_90d"] < 30000)).astype(int)
baseline_score_te = (stale * visible * df["impressions_90d"]).iloc[test_idx]

scaler = StandardScaler()
Xtr_s, Xte_s = scaler.fit_transform(Xtr_g), scaler.transform(Xte_g)
logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
logreg.fit(Xtr_s, ytr_g)
rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                             class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(Xtr_g, ytr_g)
rf_scores_te = rf.predict_proba(Xte_g)[:, 1]

print("base rate, held-out clients:", round(yte_g.mean(), 3))
for name, scores in [("baseline (w04 rule)", baseline_score_te),
                      ("random forest", rf_scores_te)]:
    print(f"{name:22s} AUC={roc_auc_score(yte_g, scores):.3f}  "
          f"P@20={precision_at_k(scores, yte_g, 20):.2f}  P@50={precision_at_k(scores, yte_g, 50):.2f}")

base rate, held-out clients: 0.511
baseline (w04 rule)    AUC=0.492  P@20=0.45  P@50=0.38
random forest          AUC=0.591  P@20=0.50  P@50=0.56


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.